In [1]:
pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 3.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.1/381.1 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/

In [2]:
from unsloth import FastLanguageModel

model_name = "unsloth/Llama-3.2-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,          # Auto FP16 / BF16
    load_in_4bit=False,  # Set True if VRAM is limited
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [3]:
def generate_outputs(prompt):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],  # only new tokens
        skip_special_tokens=True,
    )

    return response

In [4]:
pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 15.3 MB/s eta 0:00:00


In [5]:
!git clone https://github.com/ChemFoundationModels/ChemLLMBench.git

Cloning into 'ChemLLMBench'...
remote: Enumerating objects: 244, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 244 (delta 78), reused 96 (delta 43), pack-reused 87 (from 1)
Receiving objects: 100% (244/244), 4.27 MiB | 16.43 MiB/s, done.
Resolving deltas: 100% (103/103), done.


In [6]:
import openai
import random
import pandas as pd
from tqdm import tqdm
import numpy as np
from sklearn.metrics import f1_score,accuracy_score
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import DataStructs
from rdkit.Chem import rdMolDescriptors
from rdkit import Chem
import warnings
from rdkit import RDLogger
import datetime
import os
import time

In [7]:
random.seed(42)
#read bace dataset
bace = pd.read_csv("/content/ChemLLMBench/data/property_prediction/BACE.csv")
sample_size = 100
bace_sample= bace.sample(sample_size)
bace.drop(bace_sample.index, inplace = True)

In [8]:
mkdir /content/results/

In [9]:
bace_sample.to_csv("/content/results/BACE_test.csv",index = False)
bace.to_csv("/content/results/BACE_train.csv",index =False)
print(bace_sample['Class'].value_counts())

Class
0    50
1    50
Name: count, dtype: int64


In [26]:
# random sampling
def random_sample_examples(bace,sample_size):
    positive_examples = bace[bace["Class"] == 1].sample(int(sample_size/2))
    negative_examples = bace[bace["Class"] == 0].sample(int(sample_size/2))
    smiles = positive_examples["mol"].tolist() + negative_examples["mol"].tolist()

    class_label = positive_examples["Class"].tolist() + negative_examples["Class"].tolist()
    #convert 1 to "Yes" and 0 to "No"" in class_label
    class_label = ["Yes" if i == 1 else "No" for i in class_label]
    bace_examples = list(zip(smiles, class_label))
    return bace_examples

In [21]:
# scaffold sampling
def top_k_scaffold_similar_molecules(target_smiles, bace_data, k):
    #drop the target_smiles from the dataset
    bace_data = bace_data[bace_data["mol"] != target_smiles]
    molecule_smiles_list = bace_data['mol'].tolist()
    label_list = bace_data['Class'].tolist()
    label_list = ["Yes" if i == 1 else "No" for i in label_list]

    target_mol = Chem.MolFromSmiles(target_smiles)
    if target_mol is not None:
        target_scaffold = MurckoScaffold.GetScaffoldForMol(target_mol)
    else:
        print("Error: Unable to create a molecule from the provided SMILES string.")
        #drop the target_smiles from the dataset
        return None

    target_scaffold = MurckoScaffold.GetScaffoldForMol(target_mol)
    target_fp = rdMolDescriptors.GetMorganFingerprint(target_scaffold, 2)
    RDLogger.DisableLog('rdApp.warning')
    warnings.filterwarnings("ignore", category=UserWarning)
    similarities = []

    for i,smiles in enumerate(molecule_smiles_list):
        mol = Chem.MolFromSmiles(smiles)
        try:
            scaffold = MurckoScaffold.GetScaffoldForMol(mol)
            scaffold_fp = rdMolDescriptors.GetMorganFingerprint(scaffold, 2)
            tanimoto_similarity = DataStructs.TanimotoSimilarity(target_fp, scaffold_fp)
            # print(tanimoto_similarity)
            similarities.append((smiles, tanimoto_similarity,label_list[i]))
        except:
            continue
    similarities.sort(key=lambda x: x[1], reverse=True)
    top_5_similar_molecules = similarities[:k]
    return top_5_similar_molecules

In [22]:
sample_size = 2
target_smiles = "O1C[C@]2(N=C1N)c1cc(ccc1Oc1c2cc(OCC(C)C)cc1)-c1cncnc1"
random_examples = radom_sample_examples(bace_sample,sample_size)
print("randomly sampling examples", radom_sample_examples(bace_sample,sample_size))
print("scaffold sampling examples", top_k_scaffold_similar_molecules(target_smiles, bace_sample,sample_size))

randomly sampling examples [('S1(=O)(=O)N(CCCC1)c1cc(cc(NCC)c1)C(=O)NC(Cc1ccccc1)C(O)C[NH2+]C1CCc2c1cc(OC)cc2', 'Yes'), ('Clc1ccccc1-c1scc(-c2ccc(OCCC)cc2)c1CC(=O)NC(=[NH2+])N', 'No')]
scaffold sampling examples [('Fc1ncccc1-c1cc2c(Oc3c(cc(OC)cc3)C23N=C(OC3)N)cc1', 0.7777777777777778, 'No'), ('Clc1cc(F)c(cc1)-c1cc2c(Oc3c(cc(OC)cc3)C23N=C(OC3)N)cc1', 0.6941176470588235, 'No')]


In [23]:
def create_bace_prompt(input_smiles,pp_examples):
    prompt = """You are an expert chemist specializing in molecular property prediction.
Given a molecule’s SMILES string, determine whether the compound can inhibit Beta-site Amyloid Precursor Protein Cleaving Enzyme 1 (BACE1).

Base your judgment on structural features such as molecular weight, atom composition, bond types, functional groups, and overall drug-likeness relevant to Alzheimer’s disease therapeutics.

Output format (strict):
Respond with only one word — Yes if the molecule can inhibit BACE1, or No if it cannot.
Do not provide any explanation or additional text."""
    for example in pp_examples:
        prompt += f"SMILES: {example[0]}\nBACE-1 Inhibit: {example[-1]}\n"
    prompt += f"SMILES: {input_smiles}\nBACE-1 Inhibit:\n"
    return prompt

In [24]:
input_smiles = "O1C[C@]2(N=C1N)c1cc(ccc1Oc1c2cc(OCC(C)C)cc1)-c1cncnc1"
example_prompt = create_bace_prompt(input_smiles,random_examples)
print(example_prompt)

You are an expert chemist specializing in molecular property prediction.
Given a molecule’s SMILES string, determine whether the compound can inhibit Beta-site Amyloid Precursor Protein Cleaving Enzyme 1 (BACE1).

Base your judgment on structural features such as molecular weight, atom composition, bond types, functional groups, and overall drug-likeness relevant to Alzheimer’s disease therapeutics.

Output format (strict):
Respond with only one word — Yes if the molecule can inhibit BACE1, or No if it cannot.
Do not provide any explanation or additional text.SMILES: Fc1ccc(NC(=O)c2ncc(cc2)C#N)cc1[C@]1(N=C(OCC1)N)C
BACE-1 Inhibit: Yes
SMILES: O=C1N(C)C(=NC(=C1)C1CC1c1ccc(cc1)-c1ccc(cc1)C)N
BACE-1 Inhibit: No
SMILES: O1C[C@]2(N=C1N)c1cc(ccc1Oc1c2cc(OCC(C)C)cc1)-c1cncnc1
BACE-1 Inhibit:



In [27]:
sample_nums = [2, 4, 8]
modelname =  'LLAMA'
sample_methods = ['random', 'scaffold']
detail_save_folder = '/content/results/'

for sample_method in sample_methods:
    for sample_num in sample_nums:

        detail_predict_file = (
            detail_save_folder +
            'test_{}_{}_{}_{}.csv'.format('bace', modelname, sample_num, sample_method)
        )
        log_file = (
            detail_save_folder +
            'test_{}_{}_{}_{}.log'.format('bace', modelname, sample_num, sample_method)
        )

        print(detail_predict_file)
        print()

        # Load existing results if present
        if os.path.exists(detail_predict_file):
            detail_results = pd.read_csv(detail_predict_file).values.tolist()
        else:
            detail_results = []

        # Write log header
        now = datetime.datetime.now()
        date_time_str = now.strftime("%Y-%m-%d %H:%M:%S")
        with open(log_file, "a") as file:
            file.write("=" * 30 + date_time_str + "=" * 30 + "\n")

        # ---------------- RANDOM SAMPLING ----------------
        if sample_method == 'random':
            bace_examples = random_sample_examples(bace, sample_num)

            for i in tqdm(range(len(bace_sample))):
                mol = bace_sample.iloc[i]['mol']
                label = bace_sample.iloc[i]['Class']

                prompt = create_bace_prompt(mol, bace_examples)

                with open(log_file, "a") as file:
                    file.write(prompt + "\n")
                    file.write("=" * 50 + "\n")

                pred_text = generate_outputs(prompt)
                detail_results.append([mol, label, pred_text])

                if (i + 1) % 10 == 0:
                    pd.DataFrame(
                        detail_results,
                        columns=['bace_smiles', 'class_label', 'pred']
                    ).to_csv(detail_predict_file, index=False)

        # ---------------- SCAFFOLD SAMPLING ----------------
        elif sample_method == 'scaffold':
            for i in tqdm(range(len(bace_sample))):
                mol = bace_sample.iloc[i]['mol']
                label = bace_sample.iloc[i]['Class']

                bace_examples = top_k_scaffold_similar_molecules(
                    mol, bace, sample_num
                )

                prompt = create_bace_prompt(mol, bace_examples)

                with open(log_file, "a") as file:
                    file.write(prompt + "\n")
                    file.write("=" * 50 + "\n")

                pred_text = generate_outputs(prompt)
                detail_results.append([mol, label, pred_text])

                if (i + 1) % 10 == 0:
                    pd.DataFrame(
                        detail_results,
                        columns=['bace_smiles', 'class_label', 'pred']
                    ).to_csv(detail_predict_file, index=False)

        # Final save
        pd.DataFrame(
            detail_results,
            columns=['bace_smiles', 'class_label', 'pred']
        ).to_csv(detail_predict_file, index=False)


/content/results/test_bace_LLAMA_2_random.csv



100%|██████████| 100/100 [00:22<00:00,  4.48it/s]


/content/results/test_bace_LLAMA_4_random.csv



100%|██████████| 100/100 [00:21<00:00,  4.66it/s]


/content/results/test_bace_LLAMA_8_random.csv



...

100%|██████████| 100/100 [01:45<00:00,  1.06s/it]


/content/results/test_bace_LLAMA_4_scaffold.csv



100%|██████████| 100/100 [01:48<00:00,  1.09s/it]


/content/results/test_bace_LLAMA_8_scaffold.csv



100%|██████████| 100/100 [01:58<00:00,  1.18s/it]


In [ ]:
def create_bace_prompt_zero_shot(input_text):
    prompt = """You are an expert chemist, your task is to predict the property of molecule using your experienced chemical property
    prediction knowledge.\nPlease strictly follow the format, no other information can be provided. Given the SMILES string of a
    molecule, predict the molecular properties of a given chemical compound based on its structure, by analyzing wether it can
    inhibit(Yes) the Beta-site Amyloid Precursor Protein Cleaving Enzyme 1 (BACE1) or cannot inhibit(No) BACE1. Consider factors
    such as molecular weight, atom count, bond types, and functional groups in order to assess the compound's drug-likeness and its
    potential to serve as an effective therapeutic agent for Alzheimer's disease,please answer with only Yes or No. A template is
    provided in the beginning.\n"""
    prompt += f"SMILES: {input_text}\nBACE-1 Inhibit:\n"
    return prompt

In [ ]:
label = []
accs = []
f1_scores_hiv = []
epochs = 5
performance_results = []
detail_save_folder = '/content/results/zero-shot/'
few_shot_examples = (["SMILES1","Yes"],["SMILES2","No"])
paras = 0

In [ ]:
if paras < 0:
  paras += 1

# Use model_name instead of the model object to create shorter filenames
detail_predict_file = detail_save_folder + 'zero_shot_{}_{}.csv'.format('bace', model_name.replace('/', '_'))
log_file = detail_save_folder + 'zero_shot_{}_{}.log'.format('bace', model_name.replace('/', '_'))
print(detail_predict_file)
print()

# Create the directory if it doesn't exist
os.makedirs(detail_save_folder, exist_ok=True)

if os.path.exists(detail_predict_file):
  detail_results = pd.read_csv(detail_predict_file)
  #convert the column to list
  detail_results = detail_results.values.tolist()
else:
  detail_results = []

# append new date
# Get the current date and time
now = datetime.datetime.now()
# Convert the date and time to a string
date_time_str = now.strftime("%Y-%m-%d %H:%M:%S")
with open(log_file, "a") as file:
  file.write("=" * 30 + date_time_str + "=" * 30 + "\n")
para_index = 0

/content/results/zero-shot/zero_shot_bace_unsloth_Llama-3.2-3B-Instruct.csv



In [ ]:
for i in tqdm(range(0, len(bace_sample))):
  # print(para_index)
  if para_index < 0:
    para_index += 1
    continue
  example = [(bace_sample.iloc[i]['mol'],bace_sample.iloc[i]['Class'])]
  generated_results = []
  for text in example:
    prompt = create_bace_prompt_zero_shot(text[0])
    with open(log_file, "a") as file:
      file.write(prompt + "\n")
      file.write("=" * 50 + "\n")

    # print(prompt)
    pred_text = generate_outputs(prompt)
    # or smaller + exponential backoff if rate limits occur

    answer = pred_text.split("BACE-1 Inhibit:")[-1].strip()


    generated_results.append(answer)
    detail_results.append([text[0]] + [text[-1]] + [answer]) # Fixed: wrapped 'answer' in a list

    if (i + 1) % 10 == 0:
      details_df = pd.DataFrame(
        detail_results,
        columns=['bace_smiles', 'class_label', 'pred']
      )
      details_df.to_csv(detail_predict_file, index=False)
      print('save file')

  # after loop ends, save final version
details_df = pd.DataFrame(
  detail_results,
  columns=['bace_smiles', 'class_label', 'pred']
  )
details_df.to_csv(detail_predict_file, index=False)

  1%|          | 1/100 [00:00<00:13,  7.18it/s]

No


  2%|▏         | 2/100 [00:00<00:13,  7.52it/s]

No


  3%|▎         | 3/100 [00:00<00:13,  7.15it/s]

...

Yes
Yes


 99%|█████████▉| 99/100 [00:14<00:00,  6.87it/s]

No
Yes


100%|██████████| 100/100 [00:14<00:00,  6.85it/s]

No
save file


In [ ]:
!zip -r /content/results.zip /content/results/

updating: content/results/ (stored 0%)
updating: content/results/BACE_train.csv (deflated 68%)
updating: content/results/BACE_test.csv (deflated 67%)
updating: content/results/test_bace_LLAMA_4_random.csv (deflated 77%)
updating: content/results/test_bace_LLAMA_4_scaffold.csv (deflated 86%)
  adding: content/results/test_bace_LLAMA_8_random.log (deflated 64%)
  adding: content/results/test_bace_LLAMA_8_scaffold.log (deflated 96%)


You can download the zipped file `results.zip` from the file browser.

In [ ]:
data = pd.read_csv("/content/results/test_bace_LLAMA_7_random.csv")
data

,bace_smiles,class_label,pred
0,S(=O)(=O)(C)c1cc(ccc1)C1([NH2+]CC(O)C(NC(=O)C)...,0,No
1,O=C1N(C)C(=N[C@@]1(C12CC3CC(C1)CC(C2)C3)c1cccc...,0,No
2,Clc1cc2nc(n(c2cc1)C(CC(=O)NCc1cc(F)ccc1)CC)N,0,No
3,Clc1cc2CC(N=C(NC(Cc3ccccc3)C(=O)[O-])c2cc1)(C)C,0,No
4,Fc1ncccc1-c1cc(ccc1)C1(N=C(N)N(C)C1=O)c1cn(nc1...,1,No
5,Fc1cc(cc(F)c1)CC(NC(=O)C(N1CCC(NC(=O)C)(C(CC)C...,1,No
6,O=C1N(Cc2ccc(cc2)CNC(=O)NCCCC)C(NC1(CC1CCCCC1)...,0,No
7,O=C1N(C)C(=NC(=C1)C1CC1c1ccc(cc1)-c1ccc(cc1)C)N,0,No
8,S(=O)(=O)(NC1CCC([NH2+]CC(O)C(NC(=O)C)Cc2cc(F)...,0,No
9,Fc1ccc(NC(=O)c2ncc(cc2)C#N)cc1[C@]1(N=C(O[C@H]...,1,No


In [ ]:
# import shutil
# shutil.rmtree('/content/results')
